---

## **Part 1: LangChain Framework – Core Architecture (Agentic Use Case Perspective)**

### **The Problem LangChain Solves**
LLMs (like GPT-4) are great at generating text, but:
- They can’t access tools like search, calculator, or your company data.
- They forget everything between messages unless you give them memory.
- They can't reason step-by-step or make decisions dynamically.

**LangChain solves this by acting as a bridge.**

---

### **LangChain Key Components (Agentic Perspective)**

| Component | What It Does | Real Example |
|----------|--------------|--------------|
| **LLM** | Connects to GPT-4, Claude, Mistral, etc. | "Write an email to HR" |
| **PromptTemplate** | Creates reusable prompt structures | "Write a tweet about {topic}" |
| **Tool** | External actions (Search, API call, DB) | Weather lookup, PDF reader |
| **Agent** | Decides *when* and *how* to use tools | "Hmm… to answer this, I need weather + location" |
| **Memory** | Stores past conversations | Multi-turn chatbots |
| **Retriever** | Retrieves documents from a knowledge base | Vector search in RAG |
| **Chain** | Links LLM calls together in steps | Extract from PDF → Generate summary → Send email |

---

### **Visual Architecture: Simple Agentic System**

```
[User Query]
     |
     v
[LangChain Agent] --- decides --->
     |                           |
     |                          [Tool 1: Weather API]
     |                          [Tool 2: Flight Search]
     |                          [Tool 3: Calculator]
     |
[Final Answer to User]
```


---



---

## **Problem Statement**

In many real-world scenarios, users seek contextual information about the current date and time as part of their natural language queries. For example:

* *“What is today’s date?”*
* *“Can you tell me the current time before I schedule a meeting?”*

While large language models (LLMs) can generate approximate or static answers, they are **not inherently aware of real-time system information**. As a result, they may provide outdated or fabricated responses when asked about the current date or time.

To ensure accuracy, an intelligent assistant must:

1. **Understand natural language queries referring to temporal context.**
2. **Automatically invoke tools that fetch real-time system date and time.**
3. **Integrate tool outputs into a conversational, human-friendly response.**

This functionality is crucial in **productivity tools, scheduling assistants, personal reminders, and business applications**, where real-time awareness significantly enhances user experience and reliability.

---

## **Objective**

Design and implement a **LangChain-powered agent** that integrates real-time utilities for date and time. The agent should:

* **Detect** when a user query involves current date or time.
* **Automatically call the appropriate tool** (`get_current_date` or `get_current_time`).
* **Leverage LangChain’s tool-calling framework** to handle reasoning and orchestration.
* **Maintain a reasoning trace** using `agent_scratchpad` for clarity.
* **Provide responses** in a conversational and context-aware format, ensuring outputs are both precise and user-friendly.

---



In [1]:
# !pip install -U \
# langchain==0.3.27 \
# langchain-core==0.3.79 \
# langchain-community==0.3.31 \
# langchain-openai==0.3.35 \
# langchain-text-splitters==0.3.9 \
# langchainhub \
# faiss-cpu \
# sentence-transformers -q

In [2]:
# Agent components:
# 1. Brain: LLM (e.g., GPT-4, Claude 3)
# 2. Prompt Template: Guides the LLM's responses.
# 3. Tools: Functions the agent can call to perform tasks.
# 4. Memory: Stores context from previous interactions.

In [4]:
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Langchain Simple Agent

# Import required libraries

# For calling the Openai API
from langchain_openai import ChatOpenAI # LLM BRAIN - ENGINE
# # For calling the Anthropic API
# from langchain_anthropic import ChatAnthropic # LLM BRAIN - ENGINE
# # For Azure OpenAI API
# from langchain_azure import ChatAzureOpenAI # LLM BRAIN - ENGINE
# # For calling the Google API
# from langchain_google import ChatGoogle # LLM BRAIN - ENGINE
# # For calling the Cohere API
# from langchain_cohere import ChatCohere # LLM BRAIN - ENGINE
# # For calling the Hugging Face API
# from langchain_huggingface import ChatHuggingFace # LLM BRAIN - ENGINE
# # For calling the Llama 2 API
# from langchain_llama import ChatLlama # LLM BRAIN - ENGINE
# # For calling the Gemini API
# from langchain_gemini import ChatGemini # LLM BRAIN - ENGINE
# # For calling the Mistral API
# from langchain_mistral import ChatMistral # LLM BRAIN - ENGINE

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder # MESSAGES - CHASSIS
# ChatPromptTemplate: It is used to create a prompt template for chat models. This helps in structuring the input for the model.
# MessagesPlaceholder: It is used to indicate where the messages should be placed in the prompt template. This allows for dynamic insertion of messages during the conversation.

from langchain.tools import tool # HANDS - DASHBOARD
# tool: It is used to define a tool that can be called by the agent. This allows the agent to perform specific actions or retrieve information.

from langchain.agents import create_tool_calling_agent, AgentExecutor
# create_tool_calling_agent: It is used to create an agent that can call tools. This allows the agent to use the defined tools to perform tasks.
# AgentExecutor: It is used to execute the agent. This allows the agent to process inputs

In [7]:
import os
# os: It is used to interact with the operating system, such as setting environment variables.

from dotenv import load_dotenv
# load_dotenv: It is used to load environment variables from a .env file. This allows you to manage sensitive information like API keys without hardcoding them in your code.

In [8]:
# Step2: Load Environment Variables
load_dotenv()
# Load environment variables from a .env file
# This is useful for managing sensitive information like API keys without hardcoding them in your code.

True

In [9]:
llm = ChatOpenAI(model="gpt-4o")
# https://python.langchain.com/docs/integrations/chat/openai/

# ChatOpenAI(model, temperature, max_tokens, top_p, frequency_penalty, presence_penalty, stop, n, best_of, logit_bias, user, system_prompt, extra_kwargs)
# model: The name of the OpenAI model to use (e.g., "gpt-4o", "gpt-3.5-turbo").
# temperature: Controls the randomness of the output. Higher values (e.g., 0.8) make the output more random, while lower values (e.g., 0.2) make it more deterministic.
# max_tokens: The maximum number of tokens to generate in the response.
# top_p: Controls the diversity of the output by sampling from the top p% of the probability distribution.
# frequency_penalty: Controls how much to penalize new tokens based on their existing frequency in the text so far. Higher values (e.g., 0.5) make the model less likely to repeat the same lines verbatim.
# presence_penalty: Controls how much to penalize new tokens based on whether they appear in the text so far. Higher values (e.g., 0.5) make the model more likely to talk about new topics.
# stop: A list of tokens that will stop the generation when encountered.
# n: The number of responses to generate for each input message.
# best_of: The number of responses to generate and return the best (the one with the highest token logprobs per token).
# logit_bias: A dictionary that maps token IDs to bias values, which can be used to increase or decrease the likelihood of certain tokens appearing in the output.
# user: A string that can be used to identify the end-user, which can help OpenAI to monitor and detect abuse.
# system_prompt: A string that can be used to provide additional instructions or context to the model for the entire conversation.
# extra_kwargs: A dictionary of additional keyword arguments that can be passed to the OpenAI API.

In [10]:
# Step 5: Create prompt template (IMPORTANT: include input & agent_scratchpad)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a smart assistant. You can use tools whenever necessary to answer user questions."),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")  # Required
    # MessagesPlaceholder - A special placeholder used by LangChain agents to store: a. Intermediate reasoning steps, b. Tool calls, c. Tool outputs - ReAct Style (Thoughts, Actions, and Observations)
])

In [40]:
# Demo Cell
from datetime import datetime, date, time
print(datetime.now())        # current date and time
# print(datetime.utcnow())     # UTC date and time
print(date.today())          # current date only
print(date.today().isoformat())  # current date in YYYY-MM-DD format
print(datetime.now().time())   # current time only
print(datetime.now().time().isoformat())  # current time in HH:MM:SS.microseconds format

2026-03-22 20:19:32.180308
2026-03-22
2026-03-22
20:19:32.180844
20:19:32.180972


In [41]:
# Tool Definitions
from datetime import datetime, date, time

@tool
def get_current_date():
    """Returns the current date in YYYY-MM-DD format."""
    return f"Current date is: {date.today().isoformat()}"

@tool
def get_current_time():
    """Returns the current time in HH:MM:SS.microseconds format."""
    return f"Current time is: {datetime.now().time().isoformat()}"

tools = [get_current_date, get_current_time]

In [42]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [43]:
# Wrap the agent in an AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True # Set verbose=True to see the agent's reasoning and tool calls in the output
)

In [ ]:
response = agent_executor.invoke({"input": "Explain the process of photosynthesis in 5 lines."})
print(response['output'])
# print(response)

# How does the Agent work?
# 1. The user provides an input query to the agent.
# 2. The query "input" is passed to the agent's prompt template, which structures the input for the LLM.
# 3. This structured prompt is then sent to the LLM (e.g., GPT-4) for processing. LLM is good at understanding and generating human-like text based on the input it receives.
# 4. The LLM processes the input and looks for any tool calls if available. If the LLM determines that a tool is available, then it will call the tool with the appropriate parameters and get back the output from the tool. However, if the LLM determines that no tool is available, then it will see if it can answer the question itself based on its training data and the context provided in the prompt. If it can answer the question, it will generate a response directly. If it cannot answer the question, it will generate a response indicating that it cannot answer the question.
# 5. If the tool is called, the output from the tool is then passed back to the LLM as part of the agent's scratchpad (intermediate reasoning steps). The LLM can then use this information to generate a final response to the user's query.
# 6. The final response is then returned to the user. 



> Entering new AgentExecutor chain...
Photosynthesis is the process by which green plants, algae, and certain bacteria convert light energy into chemical energy in the form of glucose. It primarily occurs in the chloroplasts of plant cells, where chlorophyll absorbs sunlight. Carbon dioxide from the air and water from the soil are the reactants in this process. The light energy splits water molecules into oxygen and hydrogen, releasing oxygen as a byproduct. The hydrogen combines with carbon dioxide to form glucose, providing energy for the plant.

> Finished chain.
Photosynthesis is the process by which green plants, algae, and certain bacteria convert light energy into chemical energy in the form of glucose. It primarily occurs in the chloroplasts of plant cells, where chlorophyll absorbs sunlight. Carbon dioxide from the air and water from the soil are the reactants in this process. The light energy splits water molecules into oxygen and hydrogen, releasing oxygen as a byproduct. 

In [15]:
print(response)

{'input': 'Explain the process of photosynthesis in 5 lines.', 'output': 'Photosynthesis is the process by which green plants, algae, and certain bacteria convert light energy into chemical energy in the form of glucose. It primarily occurs in the chloroplasts of plant cells, where chlorophyll absorbs sunlight. Carbon dioxide from the air and water from the soil are the reactants in this process. The light energy splits water molecules into oxygen and hydrogen, releasing oxygen as a byproduct. The hydrogen combines with carbon dioxide to form glucose, providing energy for the plant.'}


In [16]:
response = agent_executor.invoke({"input": "Explain the process of photosynthesis in 5 bullet points."})
print(response['output'])




> Entering new AgentExecutor chain...
- **Light Absorption:** Chlorophyll, the green pigment in plant cells, captures sunlight, which provides the energy needed for photosynthesis.
- **Water Splitting:** Water molecules absorbed by the plant roots are split into oxygen, protons, and electrons through a process called photolysis.
- **Carbon Dioxide Fixation:** Carbon dioxide from the atmosphere is taken in by the plant leaves and is fixed into organic molecules in a series of reactions known as the Calvin Cycle.
- **Energy Conversion and Storage:** The energy from the sunlight is converted into chemical energy in the form of ATP and NADPH, which are used to convert carbon dioxide into glucose.
- **Oxygen Release:** As a byproduct of photolysis, oxygen is released into the atmosphere from the plant cells.

> Finished chain.
- **Light Absorption:** Chlorophyll, the green pigment in plant cells, captures sunlight, which provides the energy needed for photosynthesis.
- **Water Splitting:*

In [17]:
response = agent_executor.invoke({"input": "What is the capital of India?"})
print(response['output'])




> Entering new AgentExecutor chain...
The capital of India is New Delhi.

> Finished chain.
The capital of India is New Delhi.


In [18]:
response = agent_executor.invoke({"input": "What is the capital of New Delhi?"})
print(response['output'])




> Entering new AgentExecutor chain...
New Delhi is not a country or state that would have a capital; rather, it is itself the capital city of India.

> Finished chain.
New Delhi is not a country or state that would have a capital; rather, it is itself the capital city of India.


In [19]:
response = agent_executor.invoke({"input": "My name is Darshan and I want to become a data scientist. Please guide me with three successful career tips."})
print(response['output'])




> Entering new AgentExecutor chain...
Embarking on a career as a data scientist is an exciting and rewarding journey. Here are three tips to guide you on your path:

1. **Develop Strong Technical Skills**: 
   - **Programming:** Master programming languages such as Python and R, which are essential for data analysis and machine learning.
   - **Statistics and Mathematics:** A solid understanding of statistics, linear algebra, and calculus is crucial for developing models and interpreting data.
   - **Machine Learning and Data Analysis:** Gain proficiency in machine learning algorithms, data visualization, and data manipulation tools like Pandas, NumPy, and scikit-learn.

2. **Work on Real-world Projects**:
   - Engage in projects that require data collection, cleaning, analysis, and visualization. This practical experience is invaluable.
   - Participate in online competitions, such as Kaggle, to tackle real-world problems and learn from the community.
   - Build a portfolio to showc

In [20]:
response = agent_executor.invoke({"input": "What is the capital of New York?"})
print(response['output'])




> Entering new AgentExecutor chain...
The capital of New York is Albany.

> Finished chain.
The capital of New York is Albany.


In [21]:
response = agent_executor.invoke({"input": "Do do you recollect my name?"})
print(response['output'])




> Entering new AgentExecutor chain...
I’m sorry, but I don’t have the ability to remember personal details such as your name. Can you remind me?

> Finished chain.
I’m sorry, but I don’t have the ability to remember personal details such as your name. Can you remind me?


In [22]:
response = agent_executor.invoke({"input": "My name is Darshan and I want to become a data scientist. Please guide me with three successful career tips. At the end, give me a personalised tip, mentioning my name."})
print(response['output'])




> Entering new AgentExecutor chain...
Here are three successful career tips to guide you on your journey to becoming a data scientist:

1. **Build a Strong Foundation in Mathematics and Statistics:**
   Data science heavily relies on mathematical concepts and statistical analysis. Make sure to have a solid understanding of probability, statistics, linear algebra, and calculus. These fundamentals are crucial for data modeling and analysis.

2. **Master Programming Languages and Tools:**
   Familiarize yourself with programming languages commonly used in data science such as Python and R. Learn to work with data manipulation and analysis libraries like Pandas, NumPy, and tools for visualization like Matplotlib and Seaborn. Additionally, understanding SQL for database management is a must.

3. **Gain Practical Experience and Work on Real Projects:**
   Apply your skills by working on real-world data science projects. Participate in online competitions on platforms like Kaggle, contribut

In [23]:
response = agent_executor.invoke({"input": "Do do you recollect my name?"})
print(response['output'])




> Entering new AgentExecutor chain...
I don't have the ability to remember information between interactions, so I don't know your name. Could you please remind me?

> Finished chain.
I don't have the ability to remember information between interactions, so I don't know your name. Could you please remind me?


In [36]:
response = agent_executor.invoke({"input": "What is today's date in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22Today's date in India is 2026-03-22.

> Finished chain.
Today's date in India is 2026-03-22.


In [37]:
response = agent_executor.invoke({"input": "What is today's date in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22Today's date in India is March 22, 2026.

> Finished chain.
Today's date in India is March 22, 2026.


In [39]:
response = agent_executor.invoke({"input": "What is today's date in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22Today's date in India is March 22, 2026.

> Finished chain.
Today's date in India is March 22, 2026.


In [45]:
response = agent_executor.invoke({"input": "What is current time?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{}`


Current time is: 20:21:01.561354The current time is 20:21:01.

> Finished chain.
The current time is 20:21:01.


In [46]:
response = agent_executor.invoke({"input": "What is todays date and current time?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22
Invoking: `get_current_time` with `{}`


Current time is: 20:21:22.861481Today's date is 2026-03-22, and the current time is 20:21:22.

> Finished chain.
Today's date is 2026-03-22, and the current time is 20:21:22.


In [47]:
response = agent_executor.invoke({"input": "What is current time in 12hr format?"})
print(response['output'])



> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{}`


Current time is: 20:21:53.365629The current time in 12-hour format is 08:21:53 PM.

> Finished chain.
The current time in 12-hour format is 08:21:53 PM.


In [48]:
response = agent_executor.invoke({"input": "Is it 22nd March 2026 today?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22
Invoking: `get_current_time` with `{}`


Current time is: 20:22:28.722727Yes, today is the 22nd of March, 2026.

> Finished chain.
Yes, today is the 22nd of March, 2026.


In [49]:
response = agent_executor.invoke({"input": "Is it Sunday today?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22Yes, today is Sunday, March 22, 2026.

> Finished chain.
Yes, today is Sunday, March 22, 2026.


In [50]:
response = agent_executor.invoke({"input": "On which date is India's Independence Day celebrated every year?"})
print(response['output'])




> Entering new AgentExecutor chain...
India's Independence Day is celebrated every year on August 15th.

> Finished chain.
India's Independence Day is celebrated every year on August 15th.


In [51]:
response = agent_executor.invoke({"input": "How many days have passed since the latest India's Republic Day?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22
Invoking: `get_current_time` with `{}`


Current time is: 20:25:23.024239India's Republic Day is celebrated on January 26 each year. As today is March 22, 2026, we can calculate the days passed since January 26, 2026.

From January 26 to March 22, the number of days is 55 days.

> Finished chain.
India's Republic Day is celebrated on January 26 each year. As today is March 22, 2026, we can calculate the days passed since January 26, 2026.

From January 26 to March 22, the number of days is 55 days.


In [53]:
response = agent_executor.invoke({"input": "How many days have passed since the latest India's Independence Day? Explain step-by-step."})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22
Invoking: `get_current_time` with `{}`


Current time is: 20:27:17.603902To calculate the number of days that have passed since the latest India's Independence Day on August 15, 2025, let's follow these steps:

1. **Identifying the Dates**:
   - India's last Independence Day: August 15, 2025
   - Current date: March 22, 2026

2. **Calculate the number of days between the two dates**:
   - From August 15, 2025, to August 31, 2025: 16 days (including the 15th)
   - September 2025: 30 days
   - October 2025: 31 days
   - November 2025: 30 days
   - December 2025: 31 days
   - January 2026: 31 days
   - February 2026 (not a leap year): 28 days
   - From March 1, 2026, to March 22, 2026: 22 days

3. **Adding these up**:
   - 16 (Aug) + 30 (Sep) + 31 (Oct) + 30 (Nov) + 31 (Dec) + 31 (Jan) + 28 (Feb) + 22 (Mar) = 219 days

Therefore, 219 days have passed since the last India's Indepen

In [55]:
response = agent_executor.invoke({"input": "my session is supposed to get over at 10 PM IST today. Can you tell me how much time I have teach more??"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22
Invoking: `get_current_time` with `{}`


Current time is: 20:27:55.353522The current date is March 22, 2026, and the time is 8:27 PM IST. Your session is supposed to get over at 10 PM IST today. Therefore, you have approximately 1 hour and 33 minutes left to teach.

> Finished chain.
The current date is March 22, 2026, and the time is 8:27 PM IST. Your session is supposed to get over at 10 PM IST today. Therefore, you have approximately 1 hour and 33 minutes left to teach.


In [57]:
response = agent_executor.invoke({"input": "What is current time in India?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_time` with `{}`


Current time is: 20:30:16.954579The current time I have is in UTC (20:30:16). India follows Indian Standard Time (IST), which is UTC+5:30. So, the current time in India is approximately 02:00:16 (2 AM).

> Finished chain.
The current time I have is in UTC (20:30:16). India follows Indian Standard Time (IST), which is UTC+5:30. So, the current time in India is approximately 02:00:16 (2 AM).


In [58]:
response = agent_executor.invoke({"input": "What is current time in Pakistan?"})
print(response['output'])




> Entering new AgentExecutor chain...

Invoking: `get_current_date` with `{}`


Current date is: 2026-03-22
Invoking: `get_current_time` with `{}`


Current time is: 20:30:19.725044The current date and time is 2026-03-22, 20:30:19.725044 UTC. Pakistan is usually 5 hours ahead of UTC, so the current time in Pakistan would be approximately 01:30:19 on 2026-03-23.

> Finished chain.
The current date and time is 2026-03-22, 20:30:19.725044 UTC. Pakistan is usually 5 hours ahead of UTC, so the current time in Pakistan would be approximately 01:30:19 on 2026-03-23.
